In [8]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader('attention.pdf')
text_documents=loader.load()


from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(text_documents)

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

db=FAISS.from_documents(documents, HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))

db

/Users/gubbasainithin/Downloads/AI/langchain/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12835.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Retriever and Chain with Langchain

In [2]:
# LLM Model
from langchain_community.llms import Ollama

llama2 = Ollama(model='llama2')
llama2

Ollama()

In [3]:
# Design ChatPrompt Template
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the following question based only on the provided context. 
Think step by step before providing a detailed answer. 
I will tip you $1000 if the user finds the answer helpful. 
<context>
{context}
</context>
Question: {input}""")

In [4]:
# # # Retrieval Chain --> Retries the top entries from vector store and passes as context to LLM for better contextual result
# Chain Introduction
# Stuff document chain
from langchain.chains.combine_documents import create_stuff_documents_chain

document_chain=create_stuff_documents_chain(llama2,prompt)

In [9]:
"""
Retrievers: A retriever is an interface that returns documents given
 an unstructured query. It is more general than a vector store.
 A retriever does not need to be able to store documents, only to 
 return (or retrieve) them. Vector stores can be used as the backbone
 of a retriever, but there are other types of retrievers as well. 
 https://python.langchain.com/docs/modules/data_connection/retrievers/   
"""
retriever=db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x124147a10>)

In [10]:
"""
Retrieval chain:This chain takes in a user inquiry, which is then
passed to the retriever to fetch relevant documents. Those documents 
(and original inputs) are then passed to an LLM to generate a response
https://python.langchain.com/docs/modules/chains/
"""
from langchain.chains import create_retrieval_chain

retrieval_chain=create_retrieval_chain(retriever,document_chain)

In [11]:
response=retrieval_chain.invoke({"input":"Scaled Dot-Product Attention"})

response['answer']

'The answer to the question "Scaled Dot-Product Attention" can be summarized as follows:\n\nScaled Dot-Product Attention is a type of attention mechanism used in deep learning models, specifically in the Transformer architecture. It is called "scaled dot-product attention" because it computes the attention weights by taking the dot product of the query and key vectors, scaling the result by 1√dk, and applying a softmax function to obtain the final attention weights.\n\nThe main difference between Scaled Dot-Product Attention and other attention mechanisms is the scaling factor of 1√dk. This scaling factor is added to the dot product of the query and key vectors before applying the softmax function. The purpose of this scaling is to address the issue of large dot products in larger dimensions, which can cause the softmax function to have very small gradients. By adding the scaling factor, the dot products are reduced in magnitude, making it easier for the softmax function to compute the